# Project Ornith: Gemma Fine-Tuning with Unsloth

This notebook trains a Gemma-2 9B model using GRPO and Unsloth's highly optimized 4-bit QLoRA. It is designed to run on a free Google Colab T4 GPU without OOM errors.

**Instructions:**
1. Upload your `dataset.jsonl` file to the Colab environment.
2. Run all cells.
3. Download the resulting `lora.gguf` file to your local Mac.

In [ ]:
!pip install unsloth
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True # Use QLoRA to save VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-9b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

In [ ]:
import json
from datasets import Dataset
import re

# 1. Load Dataset
prompts = []
responses = []
with open("dataset.jsonl", "r") as f:
    for line in f:
        data = json.loads(line)
        prompts.append(data["prompt"])
        responses.append(data.get("response", ""))
        
dataset = Dataset.from_dict({"prompt": prompts, "response": responses})

# 2. Define Reward Functions (Vectorized for TRL)
def format_reward(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        score = 0.0
        if "```" in c:
            score += 0.5
        if re.search(r'\*\*.*?\*\*', c) or re.search(r'#+\s', c):
            score += 0.5
        if score == 0.0 and len(c.strip()) > 0:
            score = 0.2
        rewards.append(score)
    return rewards

def code_quality_reward(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        if "```" not in c:
            rewards.append(0.5)
            continue
        blocks = re.findall(r'```.*?\n(.*?)```', c, re.DOTALL)
        if not blocks:
            rewards.append(0.5)
            continue
        score = 0.0
        for b in blocks:
            if "def " in b or "class " in b:
                score += 0.4
            if "try:" in b or "except " in b:
                score += 0.3
        rewards.append(min(1.0, score / max(1, len(blocks))))
    return rewards

In [ ]:
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir = "outputs",
    learning_rate = 5e-6,
    num_train_epochs = 2,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    max_prompt_length = 1024,
    max_completion_length = 1024,
    num_generations = 4,
    beta = 0.1,
    logging_steps = 10,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [format_reward, code_quality_reward],
    args = training_args,
    train_dataset = dataset,
    processing_class = tokenizer,
)

trainer.train()

In [ ]:
# Export to GGUF format for Ollama
print("Exporting to GGUF...")
model.save_pretrained_gguf("lora", tokenizer, quantization_method = "f16")
print("Done! Download lora.gguf and load it into your local Ollama Modelfile.")